# C03 · Synthetic Difference-in-Differences

**Paper:** Arkhangelsky, D., Athey, S., Hirshberg, D.A., Imbens, G.W., & Wager, S. (2021). Synthetic difference-in-differences. *American Economic Review, 111*(12), 4088–4118.

**Key idea:** Reweight donor units to match pre-treatment trends AND reweight post-treatment periods to balance the control mean, then run a weighted 2×2 DiD.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from empirlab.causal import SyntheticDiD

np.random.seed(42)

## 1 · Simulate Panel DGP

- N=30 units, T=20 periods, treatment at T=14
- Unit 0 is treated with ATT=3.0
- Common factor structure: Y_it = alpha_i + beta_t + eps_it

In [ ]:
N, T, T_pre = 30, 20, 14
rng = np.random.default_rng(0)
alpha = rng.standard_normal(N)
beta  = np.cumsum(rng.standard_normal(T)) * 0.3
eps   = rng.standard_normal((N, T)) * 0.5
Y = alpha[:, None] + beta[None, :] + eps
TRUE_ATT = 3.0
Y[0, T_pre:] += TRUE_ATT
print(f"Panel shape: {Y.shape}")
print(f"True ATT: {TRUE_ATT}")

## 2 · Fit Synthetic DiD

In [ ]:
sdid = SyntheticDiD(n_boot=200, random_state=42)
sdid.fit(Y, treated_units=[0], T_pre=T_pre)
print(sdid.summary().to_string())

## 3 · Visualise Treated vs Synthetic Control

In [ ]:
omega = sdid.omega_
Y_ctrl = Y[1:]   # all control units
synth = (Y_ctrl * omega[:, None]).sum(0)
t_axis = np.arange(T)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.plot(t_axis, Y[0], lw=2, label="Treated unit", color="steelblue")
ax.plot(t_axis, synth, lw=2, ls="--", label="Synthetic control", color="tomato")
ax.axvline(T_pre - 0.5, color="gray", ls=":", label="Treatment onset")
ax.set_title("Treated vs Synthetic Control"); ax.legend(); ax.set_xlabel("Period")
ax2 = axes[1]
ax2.bar(range(len(omega)), np.sort(omega)[::-1], color='steelblue', alpha=0.7)
ax2.set_title("Donor Unit Weights (sorted)"); ax2.set_xlabel("Donor rank"); ax2.set_ylabel("omega")
plt.tight_layout(); plt.show()

## 4 · Compare Simple DiD vs Synthetic DiD

In [ ]:
Y_ctrl_mean = Y[1:].mean(0)
did_simple = ((Y[0, T_pre:].mean() - Y[0, :T_pre].mean())
              - (Y_ctrl_mean[T_pre:].mean() - Y_ctrl_mean[:T_pre].mean()))
print(f"True ATT:          {TRUE_ATT:.3f}")
print(f"Simple DiD:        {did_simple:.3f}")
print(f"SyntheticDiD ATT:  {sdid.att_:.3f}  (SE={sdid.se_:.3f})")